In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
# --- par‑box ---------------------------------------------------------------
path_box = Path("/home/loai/Documents/code/RSMLExtraction/Results/Reconstruction/measure_per_box.csv")
df_box_wid = pd.read_csv(path_box)
df_box_wid

In [ ]:
df_long = df_box_wid.melt(
    id_vars=['model', 'split', 'box', 'metric', 'time'],
    value_vars=['Prediction', 'expertized', 'before_expertized'],
    var_name='Type',
    value_name='Value'
)
df_long

In [ ]:
df_grouped = df_long.groupby(['model', 'split', 'box', 'metric', 'Type', 'time'])

# order by time
df_grouped = df_grouped.apply(lambda x: x.sort_values(by='time'))

# print the first few rows of the grouped DataFrame
df_grouped.head()

## plots

In [ ]:
# for the first model, split, box and metric. for the 3 types, plot on the same graph the evolution of the value over time

import seaborn as sns

models = df_grouped.index.get_level_values('model').unique()
splits = df_grouped.index.get_level_values('split').unique()
boxes = df_grouped.index.get_level_values('box').unique()
metrics = df_grouped.index.get_level_values('metric').unique()

print("Models disponibles :", models)
print("Splits disponibles :", splits)
print("Boxes disponibles  :", boxes)
print("Metrics disponibles:", metrics)

In [ ]:
print("Number of unique Models:", len(models))
print("Number of unique Splits:", len(splits))
print("Number of unique Boxes:", len(boxes))
print("Number of unique Metrics:", len(metrics))

In [ ]:
models_num = 0
splits_num = 1
boxes_num = 3
metrics_num = 3
boxes_num += splits_num * 5

selected_model = models[models_num]
selected_split = splits[splits_num]
selected_box = boxes[boxes_num]
selected_metric = metrics[metrics_num]

print(f"Modèle sélectionné  : {selected_model}")
print(f"Split sélectionné  : {selected_split}")
print(f"Boîte sélectionnée  : {selected_box}")
print(f"Métrique sélectionnée: {selected_metric}")

In [ ]:
# plot
import matplotlib.pyplot as plt
import seaborn as sns

# Extraction
df_temp = df_grouped.xs(
    (selected_model, selected_split, selected_box, selected_metric),
    level=('model', 'split', 'box', 'metric')
)

plt.figure(figsize=(12, 6))
sns.lineplot(
    data=df_temp,
    x='time',
    y='Value',
    hue='Type',
    marker='o'
)
plt.title(f"Evolution de la {selected_metric} pour {selected_model} - {selected_split} - {selected_box}")
plt.xlabel('Temps')
plt.ylabel(selected_metric)
plt.legend(title='Type')
plt.grid(True)
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

models = df_grouped.index.get_level_values('model').unique()
splits = df_grouped.index.get_level_values('split').unique()
boxes = df_grouped.index.get_level_values('box').unique()
metrics = df_grouped.index.get_level_values('metric').unique()


def plot_evolution(model_idx, split_idx, box_idx, metric_idx):
    model = models[model_idx]
    split = splits[split_idx]

    box = boxes[box_idx + split_idx * 5]
    metric = metrics[metric_idx]

    df_temp = df_grouped.xs(
        (model, split, box, metric),
        level=('model', 'split', 'box', 'metric')
    )

    plt.figure(figsize=(16, 8))
    sns.lineplot(
        data=df_temp,
        x='time', y='Value',
        hue='Type', marker='o'
    )
    plt.title(f"{metric} – {model} / {split} / {box}")
    plt.xlabel('Temps')
    plt.ylabel(metric)
    plt.grid(True)
    plt.tight_layout()
    plt.show()


model_slider = widgets.IntSlider(min=0, max=len(models) - 1, step=1, value=0,
                                 description='Model idx')
split_slider = widgets.IntSlider(min=0, max=len(splits) - 1, step=1, value=0,
                                 description='Split idx')
box_slider = widgets.IntSlider(min=0, max=4, step=1, value=0,
                               description='Box idx')
metric_slider = widgets.IntSlider(min=0, max=len(metrics) - 1, step=1, value=0,
                                  description='Metric idx')

ui = widgets.HBox([model_slider, split_slider, box_slider, metric_slider])
out = widgets.interactive_output(
    plot_evolution,
    {'model_idx': model_slider,
     'split_idx': split_slider,
     'box_idx': box_slider,
     'metric_idx': metric_slider}
)

display(ui, out)
